# Statistical Risk — Worst-Case Probability of Loss

**Continuous Optimization (MasterMath) — Lecture 1 demo.**

Suppose a revenue $R$ is distributed on $n$ equidistant points $r_i$ on $\Omega = [-10, 100]$. We know only some prior (moment) information:

- Mean: $\mathbb{E}[R] \in [15, 20]$
- Second moment: $\mathbb{E}[R^2] \in [500, 600]$
- Higher-order info: $\mathbb{E}[3R^3 - 2R] = 40000$

**Question.** What is the *worst* (largest) probability of a loss, $\mathbb{P}(R < 0)$, consistent with this information?

With $p_i = \mathbb{P}(R = r_i)$ this is a linear program:

$$\max_{p \ge 0}\ \sum_{i : r_i < 0} p_i \quad \text{s.t.} \quad \sum_i p_i = 1,\ \ 15 \le \sum_i r_i p_i \le 20,\ \ 500 \le \sum_i r_i^2 p_i \le 600,\ \ \sum_i (3 r_i^3 - 2 r_i) p_i = 40000.$$

In [ ]:
# Colab does not ship CVXPY by default; install it (skipped if already present).
try:
    import cvxpy  # noqa: F401
except ImportError:
    %pip install -q cvxpy

In [ ]:
import numpy as np
import cvxpy as cp

# Support: n equidistant points on [-10, 100]. (Lower n for a faster solve.)
n = 100_000
r = np.linspace(-10, 100, n)

# Decision variable: the probability mass p_i = P(R = r_i).
p = cp.Variable(n, nonneg=True)

# p is a probability distribution + the given prior (moment) information.
constraints = [
    cp.sum(p) == 1,                                     # total probability
    cp.sum(cp.multiply(r, p)) >= 15,                    # E[R] in [15, 20]
    cp.sum(cp.multiply(r, p)) <= 20,
    cp.sum(cp.multiply(r**2, p)) >= 500,                # E[R^2] in [500, 600]
    cp.sum(cp.multiply(r**2, p)) <= 600,
    cp.sum(cp.multiply(3 * r**3 - 2 * r, p)) == 40000,  # E[3R^3 - 2R] = 40000
]

# Objective: maximise the probability of a loss, P(R < 0).
objective = cp.Maximize(cp.sum(p[r < 0]))

problem = cp.Problem(objective, constraints)
problem.solve()

print(f"Status: {problem.status}")
print(f"Worst-case probability of loss  P(R < 0) = {problem.value:.4f}")

In [ ]:
# The optimal distribution concentrates on a few atoms: that is the
# worst-case scenario the data still allows.
import matplotlib.pyplot as plt

mass = p.value
support = mass > 1e-6  # the handful of points carrying probability

plt.figure(figsize=(8, 4))
plt.stem(r[support], mass[support], basefmt=" ")
plt.axvspan(r.min(), 0, color="tab:red", alpha=0.1, label="loss region $R<0$")
plt.xlabel("return $r$")
plt.ylabel("probability mass $p$")
plt.title(f"Worst-case distribution — P(R<0) = {problem.value:.3f}")
plt.legend()
plt.show()